# 11 · F.R.I.D.A.Y. watches the whole conversation

Chapter 10 scored single answers. Real problems hide in **multi-turn** behaviour: an agent that repeats the same failing action three times can score well on every individual turn. This chapter evaluates:

| Level | Data | Source type |
|---|---|---|
| **Conversation dataset** | JSONL with a `messages` array | `jsonl` |
| **Deployed conversations** 🧪 | real conversations in Application Insights, by conversation id | `azure_ai_trace_data_source_preview` + `conversation_id_source` |
| **Sampled live traffic** 🧪 | recent conversations of an agent | `trace_source: agent_filter` |

> 🧪 Trace evaluation is **preview** and needs a recent SDK (the docs name `azure-ai-projects==2.7.0` for it). Tracing must be on (chapter 03) and Application Insights ingestion takes a few minutes. Deep dive: [docs/datasets-evals.md](../docs/datasets-evals.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_evals import show_scores

import time
from azure.ai.projects.models import PromptAgentDefinition

project_openai = openai_client()   # the project's OpenAI client: conversations, responses, evals
to_clean = {"conversations": [], "evals": [], "agents": [], "datasets": []}
whole_conversation = {"messages": "{{item.messages}}"}   # conversation-level evaluators read all the messages at once

## 1 · Conversation dataset
Three prepared conversations in [conversations.jsonl](../data/conversations.jsonl): a good one, a *frustrating* one (the agent repeats itself) and a trivial one. First upload them and define an evaluation with three conversation-level evaluators. `sample=False`: there is no target, the conversations are the data.

In [ ]:
CONVERSATIONS_FILE = ROOT / "data" / "conversations.jsonl"
print(CONVERSATIONS_FILE.read_text())
conversation_dataset = project().datasets.upload_file(name="stark-conversations", version="1", file_path=str(CONVERSATIONS_FILE))
to_clean["datasets"].append(conversation_dataset.name)

conversation_schema = item_schema({"messages": {"type": "array"}}, required=["messages"], sample=False)
conversation_evaluation = project_openai.evals.create(name="stark-conversations", data_source_config=conversation_schema, testing_criteria=[
    criterion("customer_satisfaction", "customer_satisfaction", whole_conversation),
    criterion("task_completion", "task_completion", whole_conversation),
    criterion("coherence", "coherence", whole_conversation),
])
to_clean["evals"].append(conversation_evaluation.id)
print("evaluation:", conversation_evaluation.id)

### Step 1 · Score whole conversations

**Goal:** catch a failure that no single turn shows. The second conversation is polite in every turn, but the agent repeats the same action and the user gets nowhere:

| Turn | Role | Message |
|---|---|---|
| 1 | user | I need to reset my workshop badge. |
| 2 | assistant | I sent a reset link. |
| 3 | user | I did not get it. |
| 4 | assistant | I sent a reset link. |
| 5 | user | Still nothing. Can you reset it directly? |
| 6 | assistant | I sent a reset link. |

`extra_body={"evaluation_level": "conversation"}` makes the judge score each conversation as a whole. `show_scores()` ([stark_evals.py](stark_evals.py)) shows a row per conversation (its first user message), a column per evaluator, then each conversation folded and the reasons of the failed results in full.

**Expected:** ❓ judge scores, so check the reasons. Conversation 2 fails `task_completion` and `customer_satisfaction`: the badge is never reset. Conversations 1 and 3 pass both. `coherence` may pass on all three: a repeated sentence is still coherent text, which is why it alone would miss the problem.

In [ ]:
prepared_run = project_openai.evals.runs.create(eval_id=conversation_evaluation.id, name="prepared-conversations",
                                                data_source={"type": "jsonl", "source": {"type": "file_id", "id": conversation_dataset.id}},
                                                extra_body={"evaluation_level": "conversation"})
prepared_run = wait_eval_run(conversation_evaluation.id, prepared_run.id, project_openai)
print("pass rates:", pass_rates(prepared_run))
show_scores(conversation_evaluation.id, prepared_run.id, project_openai)

## 2 · Create real, traced conversations
Agent calls are traced **server-side** (chapter 04): Foundry writes the spans to Application Insights, so this chapter needs no `configure_azure_monitor`. Create an agent and two short conversations with it, two user turns each (`TRACED_TURNS`). The agent has no tools, so it can talk about an overheating suit but cannot order a coffee.

In [ ]:
AGENT_NAME = "jarvis-nb11"
JARVIS_INSTRUCTIONS = "You are J.A.R.V.I.S. Help with workshop questions. Be concise."
traced_agent = project().agents.create_version(agent_name=AGENT_NAME, definition=PromptAgentDefinition(
    model=model("chat"), instructions=JARVIS_INSTRUCTIONS))
to_clean["agents"].append(AGENT_NAME)

TRACED_TURNS = [("A suit is overheating in the hangar.", "What should I do first?"),
                ("Order me a coffee.", "Make it a double.")]
conversation_ids = []   # this run's conversations: section 3 evaluates exactly these
for turns in TRACED_TURNS:
    conversation = project_openai.conversations.create()
    conversation_ids.append(conversation.id)
    to_clean["conversations"].append(conversation.id)
    for text in turns:
        response = project_openai.responses.create(conversation=conversation.id, input=text, extra_body=agent_ref(traced_agent))
        print(f"> {text}\n{response.output_text}\n")
print(conversation_ids)

## 3 · Evaluate those conversations from Application Insights 🧪
No replay: the service reads the spans from Application Insights. Run section 2 first: this section evaluates the conversations in `conversation_ids`.

### Step 1 · Wait for the spans

The spans need 2–5 minutes to arrive, so the cell waits for them with `kql_wait`: one trace per turn, as in chapter 04, so 4 traces for 2 conversations of 2 turns.

In [ ]:
has_ids = " or ".join(f'tostring(Properties) has "{c}"' for c in conversation_ids)
kql_wait(f"AppDependencies | where TimeGenerated > ago(2h) | where {has_ids} | distinct OperationId",
         hours=2, min_rows=2 * len(conversation_ids))   # 2 turns per conversation

### Step 2 · Score the conversations by id

**Goal:** the same kind of conversation-level scores as section 1, but for conversations that really happened. The data source `azure_ai_trace_data_source_preview` with a `conversation_id_source` names them by id.

| Input | Value |
|---|---|
| conversation 1 | "A suit is overheating in the hangar." then "What should I do first?" |
| conversation 2 | "Order me a coffee." then "Make it a double." |
| evaluators | `output_quality`, `task_completion`, `customer_satisfaction` |

**Expected:** ❓ judge scores, and the answers are the model's, so this is not checked. Conversation 1 should pass: the agent can give first steps. Conversation 2 should fail `task_completion`: the agent has no tool to order anything. ❓ What the input column shows for a trace item was not checked: `show_scores()` falls back to the conversation id when an item has no `query` or `messages`.

In [ ]:
trace_criteria = [
    criterion("output_quality", "output_quality", {**whole_conversation, "tool_definitions": "{{item.tool_definitions}}"}),
    criterion("task_completion", "task_completion", whole_conversation),
    criterion("customer_satisfaction", "customer_satisfaction", whole_conversation),
]
trace_evaluation = project_openai.evals.create(name="stark-traces", data_source_config={"type": "azure_ai_source", "scenario": "traces"},
                                               testing_criteria=trace_criteria)
to_clean["evals"].append(trace_evaluation.id)

trace_run = project_openai.evals.runs.create(
    eval_id=trace_evaluation.id, name="by-conversation-id",
    data_source={"type": "azure_ai_trace_data_source_preview",
                 "trace_source": {"type": "conversation_id_source", "conversation_ids": conversation_ids}},
    extra_body={"evaluation_level": "conversation"},
)
trace_run = wait_eval_run(trace_evaluation.id, trace_run.id, project_openai)
print("pass rates:", pass_rates(trace_run))
show_scores(trace_evaluation.id, trace_run.id, project_openai)

## 4 · Sample live traffic of an agent 🧪
In production you do not know the conversation ids. Let the service pick recent ones of an agent (optionally with intelligent sampling for diversity). The run reuses `trace_evaluation` from section 3.

The cell only **starts** the run and prints its id; it does not wait. To see the scores, wait and show them like section 3:

```python
live_traffic_run = wait_eval_run(trace_evaluation.id, live_traffic_run.id, project_openai)
show_scores(trace_evaluation.id, live_traffic_run.id, project_openai)
```

❓ Which conversations `agent_filter` picks was not checked. Here it can only find conversations with `jarvis-nb11`: the ones from section 2, plus those of earlier runs of this notebook in the last 24 hours, once their spans arrived.

In [ ]:
now = int(time.time())
live_traffic_run = project_openai.evals.runs.create(
    eval_id=trace_evaluation.id, name="agent-filter-last-24h",
    data_source={"type": "azure_ai_trace_data_source_preview",
                 "trace_source": {"type": "agent_filter", "agent_name": AGENT_NAME,
                                  "start_time": now - 24 * 3600, "end_time": now + 600, "max_traces": 5}},
    extra_body={"evaluation_level": "conversation"},
)
print("started", live_traffic_run.id, "(not awaited, see above)")

## 5 · Continuous evaluation
Schedule section 4 (a nightly job, or a portal setting on the agent's **Monitor** tab) and the pass rates become a **time series**: you see quality drift when you change a prompt, a model or a tool. Pair it with alerts on Application Insights metrics (latency, error rate, tokens).

## 👀 In the portal
**Evaluation**: `stark-conversations` and `stark-traces`. The conversations they score are the ones in **Traces**.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for conversation_id in to_clean["conversations"]:
    try: project_openai.conversations.delete(conversation_id)
    except Exception: pass
for eval_id in to_clean["evals"]:
    try: project_openai.evals.delete(eval_id)
    except Exception as exc: print("skip", eval_id, type(exc).__name__)
for name in to_clean["agents"]:
    delete_agent(name)
for name in to_clean["datasets"]:
    delete_dataset(name)

## 🎯 Recap
Single answers tell you if a *response* is good. Conversations tell you if the *experience* is good. Traces tell you if *production* is good.

> 💡 Next: Act IV – the senses. [12 · OCR and Read](12_senses_ocr_read.ipynb).